

Pandas groupby notes · MD
# Pandas GroupBy — Complete Notes (Lec 23)
 
## GroupBy Kya Hai?
 
`groupby` se hum kisi column ke basis pe data ke **groups** bana sakte hain. Sawaal yeh hai ki kis type ke column pe groupby lagayein — generally columns do type ke hote hain: **numerical** aur **categorical**. Groups hamesha **categorical column** pe banate hain (jaise Genre, Director), kyunki usme distinct categories hoti hain.
 
```python
movies = pd.read_csv("imdb-top-1000.csv")
 
genres = movies.groupby('Genre')   # bracket ke andar categorical column pass karo
print(genres)   # ek GroupBy object return hota hai (khud data nahi dikhta)
```
 
GroupBy ke peeche ka poora idea **Split → Apply → Combine** strategy hai:
 
```mermaid
flowchart LR
    A["Original Data<br/>(saari movies)"] --> B["SPLIT<br/>Genre ke basis pe groups bante hain"]
    B --> C["APPLY<br/>Har group pe function lagao<br/>(sum, mean, max...)"]
    C --> D["COMBINE<br/>Saare results wapas ek table mein jud jaate hain"]
```
 
## Basic Aggregation Functions
 
GroupBy object pe seedha built-in aggregation functions laga sakte ho:
 
```python
genres.sum()                    # har group ke numeric columns ka sum, DataFrame return karta hai
genres.min()                    # har group ka min
genres.mean(numeric_only=True)  # sirf numeric columns ka mean (warna sabhi columns pe try karega)
genres.std(numeric_only=True)   # standard deviation
```
 
`numeric_only=True` isliye lagate hain kyunki agar text columns (jaise Director naam) bhi ho, to mean/std jaise functions un pe fail ho jaate ya galat result de sakte hain.
 
## Real Questions Solve Karna — Order Matters!
 
**Question:** Top 3 genres jinhone sabse zyada Gross kamaya?
 
```python
movies.groupby("Genre").sum()["Gross"].sort_values(ascending=False).head(3)
```
Yahan pehle **saare** numeric columns ka sum nikala, phir sirf `"Gross"` column extract kiya.
 
**Better tareeka** (ulta order):
```python
movies.groupby("Genre")["Gross"].sum().sort_values(ascending=False).head(3)
```
Yahan pehle hi `"Gross"` column select kar liya, phir sirf usi pe sum lagaya — matlab zaroorat se zyada calculation nahi hui.
 
```mermaid
flowchart TD
    A["Approach 1 (kam efficient)"] --> A1["groupby → sum() saare columns pe"] --> A2["phir Gross column extract"]
    B["Approach 2 (better)"] --> B1["groupby → pehle Gross column select"] --> B2["phir sirf usi pe sum()"]
    A2 --> Z["Same result, lekin extra kaam hua"]
    B2 --> Y["Same result, kam calculation"]
```
 
Doosra tareeka behtar hai kyunki pehle hi target column select kar li, baaki columns pe fazool calculation nahi hui.
 
**Aur bhi examples isi pattern pe:**
```python
# Sabse zyada average IMDB rating wala genre
movies.groupby("Genre")["IMDB_Rating"].mean().sort_values(ascending=False).head(1)
 
# No_of_Votes ko popularity metric maankar sabse popular director
movies.groupby("Director")["No_of_Votes"].sum().sort_values(ascending=False).head(1)
 
# Har genre ki highest rated movie ki rating
movies.groupby("Genre")["IMDB_Rating"].max()
 
# Har actor (Star1) ne kitni movies ki hain
movies.groupby("Star1")["Series_Title"].count().sort_values(ascending=False)
```
 
> Note: "har genre ki highest **rated movie**" (poori row chahiye, sirf rating nahi) ke liye simple aggregation kaafi nahi — uske liye aage `apply()` lagega.
 
## GroupBy Object — Attributes & Methods
 
| Attribute/Method | Kaam |
|---|---|
| `len(genres)` | Kitne groups bane hain |
| `movies["Genre"].nunique()` | Same jaankari — kitni unique categories hain |
| `genres.size()` | Har group mein kitni rows hain (Series return karta hai) |
| `movies["Genre"].value_counts()` | Same jaankari, descending sorted |
| `genres.first()` | Har group ka pehla item/row |
| `genres.last()` | Har group ka aakhri item/row |
| `genres.nth(6)` | Har group ka nth row (0-indexed, isliye 7th chahiye to `nth(6)`) |
| `genres.get_group("Horror")` | Ek particular group ka poora data (jaise `movies[movies["Genre"]=="Horror"]` boolean masking) |
| `genres.groups` | Dictionary return karta hai — key = group ka naam, value = uske indexes |
 
## describe() / sample() on Groups
 
```python
genres.describe()          # har group ke har numeric column ki mathematical summary
genres.sample()            # har group se ek random row
genres.sample(2, replace=True)   # har group se 2 random rows (replace=True: agar kisi group mein 2 se kam rows hain to woh repeat ho jaayengi)
genres.nunique()           # har group mein har column ki unique counts
```
 
## `agg()` — Multiple Aggregations Ek Saath
 
Sabhi numeric columns pe ek hi aggregation function (jaise `sum()`) lagana simple hai lekin hamesha smart nahi — kabhi kabhi alag-alag column pe alag function chahiye hota hai.
 
**Dictionary style** — har column ke liye specific function:
```python
genres.agg({
    'Runtime': 'mean',
    'IMDB_Rating': 'mean',
    'No_of_Votes': 'sum',
    'Gross': 'sum',
    'Metascore': 'min'
})
```
 
**List style** — saare selected columns pe multiple functions:
```python
genres.agg(['min', 'max', 'mean'])
```
Isse har column ke liye 3 alag columns ban jaate hain (min, max, mean). Lekin problem yeh hai ki `min`/`max` text columns (jaise Director naam) pe bhi chal jaate hain, jabki `mean` sirf numeric data pe chalta hai — is confusion se bachne ke liye pehle numeric columns select kar lo:
 
```python
numeric_columns = movies.select_dtypes(include="number").columns
genres[numeric_columns].agg(["min", "max", "mean"])
```
(GroupBy object pe direct `select_dtypes` nahi chalta, isliye original DataFrame se columns nikal ke phir groupby object pe apply karte hain.)
 
```mermaid
flowchart TD
    A["agg() ka use"] --> B["Dict style<br/>{'col1':'func1', 'col2':'func2'}<br/>har column ka apna function"]
    A --> C["List style<br/>['min','max','mean']<br/>saare selected columns pe sabhi functions"]
    C --> D["Pehle numeric_columns select karo<br/>warna text columns pe bhi chalega"]
```
 
## Looping on Groups
 
```python
for group, data in genres:   # group = naam (string), data = uska DataFrame
    print(data)
```
 
Isi looping concept se woh sawaal solve hota hai jo pehle chhoda tha — **har genre ki highest rated movie (poori row)**:
 
```python
movies_list = []
for group, data in genres:
    highest_rated = data[data['IMDB_Rating'] == data['IMDB_Rating'].max()]
    movies_list.append(highest_rated)
 
df = pd.concat(movies_list)   # DataFrame pe direct append nahi chalta, isliye list mein jama karke concat karte hain
```
 
## Split → Apply → Combine ka Real Use: `apply()`
 
`apply()` function har group ke andar ghuskar tumhara custom function chala deta hai, phir results ko combine kar deta hai.
 
```mermaid
flowchart LR
    A["genres.apply(function)"] --> B["Har group (DataFrame) function ko milta hai"]
    B --> C["Function apna logic chalata hai<br/>(naya column banao, calculate karo, filter karo)"]
    C --> D["Saare groups ke results combine hoke wapas milte hain"]
```
 
**Example — har group mein "A" se shuru hone waali movies ka count:**
```python
def foo(group):
    return group["Series_Title"].str.startswith("A").sum()
 
genres.apply(foo)
```
`.str.startswith("A")` ek boolean Series deta hai (True/False har row ke liye), `.sum()` se True ki ginti mil jaati hai.
 
**Example — har movie ki apne genre ke andar ranking (IMDB rating ke basis pe):**
```python
def ranking(group):
    group["genre_rank"] = group["IMDB_Rating"].rank(ascending=False)
    return group
 
result = genres.apply(ranking)
```
 
**Example — Normalized IMDB rating (group-wise):**
 
Formula: `(rating - group_min) / (group_max - group_min)`
 
```python
def normalized_rating(group):
    group["norm_rating"] = (group["IMDB_Rating"] - group["IMDB_Rating"].min()) / \
                            (group["IMDB_Rating"].max() - group["IMDB_Rating"].min())
    return group
 
result = genres.apply(normalized_rating)
result["norm_rating"] = result["norm_rating"].fillna(1.0)   # jahan max==min wahan 0/0 hota hai, use 1.0 se fill karte hain
```
 
## GroupBy on Multiple Columns
 
List pass karke ek se zyada column ke basis pe groups bana sakte ho:
 
```python
duo = movies.groupby(["Director", "Star1"])
duo.size()
 
duo.get_group(("Aamir Khan", "Amole Gupte"))   # tuple pass karna padta hai
```
 
**Sabse zyada kamaane wala director-actor combo:**
```python
duo["Gross"].max().sort_values(ascending=False).head(1)   # galat — ek movie ka max le raha hai
duo["Gross"].sum().sort_values(ascending=False).head(1)   # sahi — total kamai chahiye thi
```
 
**Actor-Genre combo jo avg Metascore ke terms mein best hai:**
```python
duo1 = movies.groupby(["Star1", "Genre"])
duo1["Metascore"].mean().reset_index().sort_values("Metascore", ascending=False)
```
 
**Multiple columns pe multiple aggregations:**
```python
numeric_cols = movies.select_dtypes(include="number").columns
duo[numeric_cols].agg(["min", "max", "mean"])
```
 
## Real-World Practice — IPL Ball-by-Ball Data
 
```python
ipl = pd.read_csv("deliveries.csv")   # 2022 tak ke saare IPL matches ka ball-by-ball data
```
 
```python
# Top 10 batsman (total runs ke basis pe)
batter = ipl.groupby("batsman")
batter["batsman_runs"].sum().sort_values(ascending=False).head(10)
 
# Sabse zyada sixes maarne wala batsman
sixes_only = ipl[ipl["batsman_runs"] == 6]
sixes_only.groupby("batsman")["batsman"].count().sort_values(ascending=False).head(1).index[0]
 
# Last 5 overs (over > 15) mein sabse zyada 4s+6s maarne wala batsman
fours_sixes = ipl[((ipl["batsman_runs"]==6) | (ipl["batsman_runs"]==4)) & (ipl["over"]>15)]
batter1 = fours_sixes.groupby("batsman")
batter1["batsman_runs"].count().sort_values(ascending=False).head(1).index[0]
 
# Virat Kohli ka har bowling team ke against total runs
record = (ipl[ipl["batsman"]=="V Kohli"]).groupby("bowling_team")
record["batsman_runs"].sum()
 
# Kisi bhi batsman ka highest individual score (match-wise total nikaal ke)
def highest(batsman):
    temp_df = ipl[ipl['batsman'] == batsman]
    return temp_df.groupby('match_id')['batsman_runs'].sum().sort_values(ascending=False).head(1).values[0]
 
highest('DA Warner')
```
 
---
 
## Quick Recap
 
```mermaid
flowchart TD
    A["groupby('col')"] --> B["Built-in Aggregations<br/>(sum, mean, min, max, std)"]
    A --> C["GroupBy Methods<br/>(size, first, last, nth, get_group, groups)"]
    A --> D["agg() — dict ya list style<br/>multiple functions ek saath"]
    A --> E["Looping<br/>for group, data in obj"]
    A --> F["apply() — Split→Apply→Combine<br/>custom function har group pe"]
    A --> G["Multiple columns pe groupby<br/>groupby([col1, col2])"]
```
 
**GroupBy = pehle categorical column ke basis pe data split karo, phir har group pe kuch calculate/apply karo, phir sab combine ho jaata hai.** Simple aggregation (`sum`, `mean`) built-in functions se ho jaata hai; jab row-level ya custom logic chahiye ho (jaise ranking, normalization, poori row nikaalna) tab `apply()` ka use hota hai.
 




In [ ]:
import pandas as pd
import numpy as np

movies = pd.read_csv(r"C:\Users\Asus\Downloads\imdb-top-1000.csv")

In [22]:
#groupby main kisi column ke basis  pe  main jo hai groupp form karta hoon 
#par main kis type ke column ka istemaal karunga groups form karne ke liye iska idea kasie lagega ?
#generally 2 types ke column honge numerical  with numeric data adn categorical kyunki isme category hai to group by hamesha categorical column main lagega 

#Ab suppose  jitnee different genres hai uske bases pe data ke groupps form karna chata hoon 
genres = movies.groupby('Genre') #bracket ke andaar categorical column pass karunga 
print(genres)
#movies.groupby('Genre') ye ek object return karega 

In [ ]:
# Applying builtin aggregation fuctions on groupby objects
genres.sum() #Dataframe return karega jisme index will be categorical column and original data ke numerical columns ko pakadne group wise un numerical value ka sum rahe hain 

genres.min() #har groups main se min nikalke dataframe main  daalke  de dega 

,Series_Title,Released_Year,Runtime,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
Genre,,,,,,,,,
Action,300,1924,45,7.6,Abhishek Chaubey,Aamir Khan,25312,3296.0,33.0
Adventure,2001: A Space Odyssey,1925,88,7.6,Akira Kurosawa,Aamir Khan,29999,61001.0,41.0
Animation,Akira,1940,71,7.6,Adam Elliot,Adrian Molina,25229,128985.0,61.0
Biography,12 Years a Slave,1928,93,7.6,Adam McKay,Adrien Brody,27254,21877.0,48.0
Comedy,(500) Days of Summer,1921,68,7.6,Alejandro G. Iñárritu,Aamir Khan,26337,1305.0,45.0
Crime,12 Angry Men,1931,80,7.6,Akira Kurosawa,Ajay Devgn,27712,6013.0,47.0
Drama,1917,1925,64,7.6,Aamir Khan,Abhay Deol,25088,3600.0,28.0
Family,E.T. the Extra-Terrestrial,1971,100,7.8,Mel Stuart,Gene Wilder,178731,4000000.0,67.0
Fantasy,Das Cabinet des Dr. Caligari,1920,76,7.9,F.W. Murnau,Max Schreck,57428,337574718.0,NaN


In [24]:
genres.mean(numeric_only=True)  #isse sirf har group ke numeric columns ka mean niklega elsse yeh default to  dsaare groups ke saare columns ka mean nikalne ki koshsihs karega 

,Runtime,IMDB_Rating,No_of_Votes,Gross,Metascore
Genre,,,,,
Action,129.046512,7.949419,420246.581395,1.897224e+08,73.419580
Adventure,134.111111,7.937500,313557.819444,1.319017e+08,78.437500
Animation,99.585366,7.930488,268032.073171,1.784326e+08,81.093333
Biography,136.022727,7.938636,272805.045455,9.404952e+07,76.240506
Comedy,112.129032,7.901290,178195.658065,1.010572e+08,78.720000
Crime,126.392523,8.016822,313398.271028,7.899656e+07,77.080460
Drama,124.737024,7.957439,212343.612457,1.225259e+08,79.701245
Family,107.500000,7.800000,275610.500000,2.195553e+08,79.000000
Fantasy,85.000000,8.000000,73111.000000,3.913633e+08,NaN


In [25]:
genres.std(numeric_only=True)

,Runtime,IMDB_Rating,No_of_Votes,Gross,Metascore
Genre,,,,,
Action,28.500706,0.304258,432946.814748,2.256724e+08,12.421252
Adventure,33.317320,0.229781,301188.347642,1.697543e+08,12.345393
Animation,14.530471,0.253221,262173.231571,2.091840e+08,8.813646
Biography,25.514466,0.267140,271284.191372,1.363251e+08,11.028187
Comedy,22.946213,0.228771,188653.570564,1.946513e+08,11.829160
Crime,27.689231,0.335477,373999.730656,1.571191e+08,13.099102
Drama,27.740490,0.267229,305554.162841,2.201164e+08,12.744687
Family,10.606602,0.000000,137008.302816,3.048412e+08,16.970563
Fantasy,12.727922,0.141421,22179.111299,7.606861e+07,NaN


In [26]:
#find the top 3 genres jisne sabse zyada kamai  kari hai movies 

movies.groupby("Genre").sum()["Gross"]
#isse hoga yeh ki sabse pehle geres ke basis pe alag groups bana diya ab har group mainn toal kamai ke liye sum l= kar diya jisse saaare ke saare group ke numeric columns ke data add hogaye then ["Gross"] laga diya taaki specifically gross column se kaam hai to usko extract kar pau 

# return a series with inindexa scateggorical column and values 

#Par ye data jo hai woh index ke basis pe sorted hai to main isko value s ke basis pe sort krunga  

Genre
Action       3.263226e+10
Adventure    9.496922e+09
Animation    1.463147e+10
Biography    8.276358e+09
Comedy       1.566387e+10
Crime        8.452632e+09
Drama        3.540997e+10
Family       4.391106e+08
Fantasy      7.827267e+08
Film-Noir    1.259105e+08
Horror       1.034649e+09
Mystery      1.256417e+09
Thriller     1.755074e+07
Western      5.822151e+07
Name: Gross, dtype: float64

In [27]:
movies.groupby("Genre").sum()["Gross"].sort_values(ascending=False).head(3)

# ab upar ke teen chahiye to head(3) laga do 


Genre
Drama     3.540997e+10
Action    3.263226e+10
Comedy    1.566387e+10
Name: Gross, dtype: float64

In [28]:
#Purane question ko ab main pure ulte tareeke se bhi solve karunga 
movies.groupby("Genre")["Gross"].sum().sort_values(ascending=False).head(3)

Genre
Drama     3.540997e+10
Action    3.263226e+10
Comedy    1.566387e+10
Name: Gross, dtype: float64

In [29]:
#Ab in do tareeko main jinme hamne order change kiya  isme shi ya better kkounsa hai ?

#2nd wala kyunki pehle main zyada columns ka sum nikal rahe dusre wale main targeted ya kaam ka nikalke sirf uspe sum laga rhe

In [30]:
#GEnre with highest imdb rating average(imdb ratings ka avreage nikalo jis genre ka max hoga wo dedo)

movies.groupby("Genre")["IMDB_Rating"].mean().sort_values(ascending=False).head(1)

#YAhan sirf numeric data tha to numeric_only parameter ki zarurat nhi padi 

Genre
Western    8.35
Name: IMDB_Rating, dtype: float64

In [31]:
#No of votes ko popularity ka metric mante hue kounsa director sabse popular hai ?
movies.groupby("Director")["No_of_Votes"].sum().sort_values(ascending=False).head(1)

Director
Christopher Nolan    11578345
Name: No_of_Votes, dtype: int64

In [32]:
#Find highest rated movie of each genre 
movies
movies.groupby("Genre")["IMDB_Rating"].max()
#Har genres ki highest rating aa jayeggi 
#Abhi solution chodh kyunki multiple  cheezein lagengii kyunki aage batayenge 

Genre
Action       9.0
Adventure    8.6
Animation    8.6
Biography    8.9
Comedy       8.6
Crime        9.2
Drama        9.3
Family       7.8
Fantasy      8.1
Film-Noir    8.1
Horror       8.5
Mystery      8.4
Thriller     7.8
Western      8.8
Name: IMDB_Rating, dtype: float64

In [33]:
#find number of movies done by each actor with the help of groupby

movies.groupby("Star1")["Series_Title"].count().sort_values(ascending=False)

#Count() to rows count karega 


Star1
Tom Hanks               12
Robert De Niro          11
Clint Eastwood          10
Al Pacino               10
Humphrey Bogart          9
                        ..
Zbigniew Zamachowski     1
Zooey Deschanel          1
Çetin Tekindor           1
Éric Toledano            1
Aaron Taylor-Johnson     1
Name: Series_Title, Length: 660, dtype: int64

In [34]:
#groupby atttributes and methods

#Ab suppose jaise genres ke groups  banaye toh kitne groups forms hue to yeh  kaise bataunga ?
#Either len or nunique

len(movies.groupby("Genre"))

movies["Genre"].nunique()

#har group le andar kitne rows gaye hain ?
movies.groupby("Genre").size() #returns a series with row count of each group 

#EK aur tareeka hai value_counts()
movies["Genre"].value_counts() #bas descending sorted result deta 

Genre
Drama        289
Action       172
Comedy       155
Crime        107
Biography     88
Animation     82
Adventure     72
Mystery       12
Horror        11
Western        4
Film-Noir      3
Fantasy        2
Family         2
Thriller       1
Name: count, dtype: int64

In [35]:
#Har group ka first item nikal skte ho 
genres=movies.groupby("Genre")

genres.first() #har genre ki  pehli movie ya har group ka pehla item 
genres.last() #vice versa of above function

#HAr group ka nth item ya row do  to 
genres.nth(6) # jo dikhana hai na wo number se 1 minus kare dena mujhe 7th dataa chahiye tha  

#PArticular group ka data jaise horror genre ki saari movies 
genres.get_group("Horror") 
#boolean masking ka use like movies[movies["Genre"]=="Horror"]


,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
49,Psycho,1960,109,Horror,8.5,Alfred Hitchcock,Anthony Perkins,604211,32000000.0,97.0
75,Alien,1979,117,Horror,8.4,Ridley Scott,Sigourney Weaver,787806,78900000.0,89.0
271,The Thing,1982,109,Horror,8.1,John Carpenter,Kurt Russell,371271,13782838.0,57.0
419,The Exorcist,1973,122,Horror,8.0,William Friedkin,Ellen Burstyn,362393,232906145.0,81.0
544,Night of the Living Dead,1968,96,Horror,7.9,George A. Romero,Duane Jones,116557,89029.0,89.0
707,The Innocents,1961,100,Horror,7.8,Jack Clayton,Deborah Kerr,27007,2616000.0,88.0
724,Get Out,2017,104,Horror,7.7,Jordan Peele,Daniel Kaluuya,492851,176040665.0,85.0
844,Halloween,1978,91,Horror,7.7,John Carpenter,Donald Pleasence,233106,47000000.0,87.0
876,The Invisible Man,1933,71,Horror,7.7,James Whale,Claude Rains,30683,298791505.0,87.0
932,Saw,2004,103,Horror,7.6,James Wan,Cary Elwes,379020,56000369.0,46.0


In [36]:
genres.groups #willreturn a dictionary jahan key will be ggroup ka naam  keys will be indexes 

{'Action': [2, 5, 8, 10, 13, 14, 16, 29, 30, 31, 39, 42, 44, 55, 57, 59, 60, 63, 68, 72, 106, 109, 129, 130, 134, 140, 142, 144, 152, 155, 160, 161, 166, 168, 171, 172, 177, 181, 194, 201, 202, 216, 217, 223, 224, 236, 241, 262, 275, 294, 308, 320, 325, 326, 331, 337, 339, 340, 343, 345, 348, 351, 353, 356, 357, 362, 368, 369, 375, 376, 390, 410, 431, 436, 473, 477, 479, 482, 488, 493, 496, 502, 507, 511, 532, 535, 540, 543, 564, 569, 570, 573, 577, 582, 583, 602, 605, 608, 615, 623, ...], 'Adventure': [21, 47, 93, 110, 114, 116, 118, 137, 178, 179, 191, 193, 209, 226, 231, 247, 267, 273, 281, 300, 301, 304, 306, 323, 329, 361, 366, 377, 402, 406, 415, 426, 458, 470, 497, 498, 506, 513, 514, 537, 549, 552, 553, 566, 576, 604, 609, 618, 638, 647, 675, 681, 686, 692, 711, 713, 739, 755, 781, 797, 798, 851, 873, 884, 912, 919, 947, 957, 964, 966, 984, 991], 'Animation': [23, 43, 46, 56, 58, 61, 66, 70, 101, 135, 146, 151, 158, 170, 197, 205, 211, 213, 219, 229, 230, 242, 245, 246, 270, 33

In [37]:
#describe/sample/
genres.describe() #har group ka har numerical column ke basis pe study (mathematical summarization)

genres.sample() #random movie selection
#Man mutabik random data  chahie ho jaise har group se 2 movies 
genres.sample(2,replace=True) #Agar koi aisa genre hai jisme 2 se kam hhi movies hai  to un rows ko replace kar dega 

genres.nunique() #har group main unique counts

,Series_Title,Released_Year,Runtime,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
Genre,,,,,,,,,
Action,172,61,78,15,123,121,172,172,50
Adventure,72,49,58,10,59,59,72,72,33
Animation,82,35,41,11,51,77,82,82,29
Biography,88,44,56,13,76,72,88,88,40
Comedy,155,72,70,11,113,133,155,155,44
Crime,106,56,65,14,86,85,107,107,39
Drama,289,83,95,14,211,250,288,287,52
Family,2,2,2,1,2,2,2,2,2
Fantasy,2,2,2,2,2,2,2,2,0


In [38]:
#agg methods lets you aplly mutliple agrregation functions on groups 
genres.sum()
#But agar har numerical column pe ek aggregation fucntion lagana is not smart it simple but not good agar runtime ka avg milta to acha hota to heres how you can do that

,Series_Title,Released_Year,Runtime,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
Genre,,,,,,,,,
Action,The Dark KnightThe Lord of the Rings: The Retu...,2008200320102001200219991980197719621954200019...,22196,1367.3,Christopher NolanPeter JacksonChristopher Nola...,Christian BaleElijah WoodLeonardo DiCaprioElij...,72282412,3.263226e+10,10499.0
Adventure,InterstellarBack to the FutureInglourious Bast...,2014198520091981196819621959201319751963194819...,9656,571.5,Christopher NolanRobert ZemeckisQuentin Tarant...,Matthew McConaugheyMichael J. FoxBrad PittJürg...,22576163,9.496922e+09,5020.0
Animation,Sen to Chihiro no kamikakushiThe Lion KingHota...,2001199419882016201820172008199719952019200920...,8166,650.3,Hayao MiyazakiRoger AllersIsao TakahataMakoto ...,Daveigh ChaseRob MinkoffTsutomu TatsumiRyûnosu...,21978630,1.463147e+10,6082.0
Biography,Schindler's ListGoodfellasHamiltonThe Intoucha...,1993199020202011200220171995198420182013201320...,11970,698.6,Steven SpielbergMartin ScorseseThomas KailOliv...,Liam NeesonRobert De NiroLin-Manuel MirandaÉri...,24006844,8.276358e+09,6023.0
Comedy,GisaengchungLa vita è bellaModern TimesCity Li...,2019199719361931200919641940200120001973196019...,17380,1224.7,Bong Joon HoRoberto BenigniCharles ChaplinChar...,Kang-ho SongRoberto BenigniCharles ChaplinChar...,27620327,1.566387e+10,9840.0
Crime,The GodfatherThe Godfather: Part II12 Angry Me...,1972197419571994200219991995199120192006199519...,13524,857.8,Francis Ford CoppolaFrancis Ford CoppolaSidney...,Marlon BrandoAl PacinoHenry FondaJohn Travolta...,33533615,8.452632e+09,6706.0
Drama,The Shawshank RedemptionFight ClubForrest Gump...,1994199919941975202019981946201420061998198819...,36049,2299.7,Frank DarabontDavid FincherRobert ZemeckisMilo...,Tim RobbinsBrad PittTom HanksJack NicholsonSur...,61367304,3.540997e+10,19208.0
Family,E.T. the Extra-TerrestrialWilly Wonka & the Ch...,19821971,215,15.6,Steven SpielbergMel Stuart,Henry ThomasGene Wilder,551221,4.391106e+08,158.0
Fantasy,Das Cabinet des Dr. CaligariNosferatu,19201922,170,16.0,Robert WieneF.W. Murnau,Werner KraussMax Schreck,146222,7.827267e+08,0.0


In [39]:
genres.agg(
    {
        'Runtime':'mean',
        'IMDB_Rating':'mean',
        'No_of_Votes':'sum',
        'Gross':'sum',
        'Metascore':'min'
    }
)

,Runtime,IMDB_Rating,No_of_Votes,Gross,Metascore
Genre,,,,,
Action,129.046512,7.949419,72282412,3.263226e+10,33.0
Adventure,134.111111,7.937500,22576163,9.496922e+09,41.0
Animation,99.585366,7.930488,21978630,1.463147e+10,61.0
Biography,136.022727,7.938636,24006844,8.276358e+09,48.0
Comedy,112.129032,7.901290,27620327,1.566387e+10,45.0
Crime,126.392523,8.016822,33533615,8.452632e+09,47.0
Drama,124.737024,7.957439,61367304,3.540997e+10,28.0
Family,107.500000,7.800000,551221,4.391106e+08,67.0
Fantasy,85.000000,8.000000,146222,7.827267e+08,NaN


In [40]:
#suppose har column pe multiple aggregationn functions lagane ho toh ?
#List ka  use 
# passing list
genres.agg(['min','max','mean'])
#ye to ek group saare columns ko dharega usnke teen columns banayega min max avaerage calculatee karega par ek problems hai ki  agar isko use karna hai toh max and min str pe lag jayege par mean sirf numeric pe lagega and use bhi aggregationn fucntion hota hai mostly for numeric data toh pehle sleect karna padega 
# Panda's ka agg() function hume ek sath kai saare operations call karne ki azadi deta hai. Jab aap list ke roop mein ['min', 'max', 'mean'] pass karte hain, to Pandas har ek select hue column ke liye teen alag-alag columns bana deta hai (ek minimum ke liye, ek maximum ke liye, aur ek average ke liye).



TypeError: dtype 'str' does not support operation 'mean'

In [ ]:
#TOh hum pehle numeric columns ki list nikalenge orignial wale se since group by object main select_dtypes nhi hota 

numeric_columns=movies.select_dtypes(include="number").columns

genres[numeric_columns].agg(["min","max","mean"])

In [ ]:
#Looping onn groups

for group,data in genres: #returns a group object whose type is string and data  
   print(data)
   print() #saare groups ka data 

In [ ]:
#Ab jo question chodha tha woh karenge 
#Find highest rated movie of each genre 
df=pd.DataFrame(columns=movies.columns)

for group,data in genres:
    print(data[data["IMDB_Rating"]==data["IMDB_Rating"].max()])
#AB hamne loop chalaya thnen humko ek dataframe mila  jo us particular rgroup kii movies contain karega  then data imdb rating .max har group ki max rating degi jab sab ratings ko compare karenge toh jo bacha wohi milega yni har genre ki highest rated movie  

#Ab ye code kyunki dataframe pe append nhi chalta 
# 1. Pehle ek khali list (baksā) banate hain data jama karne ke liye
movies_list = []

# 2. Ye wahi loop hai jo Sir ne chalaya tha
for group, data in genres:
    # Har group ki sabse highest rated movie nikal li
    highest_rated = data[data['IMDB_Rating'] == data['IMDB_Rating'].max()]
    
    # Is highest rated movie ko apni list mein daal diya
    movies_list.append(highest_rated)

# 3. Last mein saari jama ki hui movies ko ek table (DataFrame) mein badal diya
df = pd.concat(movies_list)

# 4. Ab output dekhte hain
df

In [ ]:
#GRoup by ka use karke data ko group wise split kiya thenn apply function ka use karke hum kuch functionality apply karte hai ya groups pe kuch function karte hai and then we combine 

#this is split,apply,combine stratergy
genres.apply(max)  #har group main ghusega and min value extract karke dega 

In [ ]:
# split->apply->combine 
#har particular genre main kitne no of movies hai jo A se chalu hoti hai 

def foo(group):
    # group["Series_Title"].str.startswith("A")
    #   isse hoga yeh ki group ke series title ke column ko convert kiya  hai  string main and then check kiya ki us column ke data ka start a se ho raha hai ki nhi ek boolean sereis return karega jisme jo start hoag wo true and jo nhi hoga wo false
    print(group["Series_Title"].str)
    #Saare column ke trues ko add karke bata dega kio har group (genre) main har gere ki kiti movies a se start hai 

    return group
#ye sirf ek exmaple hai to see how a groupby object is converted into string example solved aage hai 


genres.apply(foo)


In [ ]:
# split->apply->combine 
#har particular genre main kitne no of movies hai jo A se chalu hoti hai 

def foo(group):
    # group["Series_Title"].str.startswith("A")
    #   isse hoga yeh ki group ke series title ke column ko convert kiya  hai  string main and then check kiya ki us column ke data ka start a se ho raha hai ki nhi ek boolean sereis return karega jisme jo start hoag wo true and jo nhi hoga wo false (boolean series for every group  )
    sum=group["Series_Title"].str.startswith("A").sum()
    #Saare column ke trues ko add karke bata dega kio har group (genre) main har gere ki kiti movies a se start hai 

    return sum

genres.apply(foo)

In [ ]:
# har movie ke samne ek  ranking dena hai and wo ranking batayega ki uske khud ke group main kitni ranking hai based on IMDB score /rating 

def ranking(group):
    group["genre_rank"]=group["IMDB_Rating"].rank(ascending=False) 
    return group


result=genres.apply(ranking) 
result[['Series_Title', 'IMDB_Rating', 'genre_rank']]

In [47]:
#FInd normalized ImDB rating (normalized rating) group wise 

#Normalizationn kaise karte hai ?
# har movie ka kuch rating hai  suppose n rating shai overall ek groupp main and uss group main ek particular movie ka normalization nikalna hai to 

#LEt us movie ka rating be x sor normalization will will x-(min rating)/(maxrating-min rating )  asia har movie ke saath hoga 

def normalized_rating(group):
    group["norm_rating"]=(group["IMDB_Rating"]-group["IMDB_Rating"].min())/(group["IMDB_Rating"].max()-group["IMDB_Rating"].min())

    return group

result=genres.apply(normalized_rating)
result["norm_rating"]=result["norm_rating"].fillna(1.0)
result


Series_Title Released_Year  \
Genre                                                                           
Action   2                                      The Dark Knight          2008   
         5        The Lord of the Rings: The Return of the King          2003   
         8                                            Inception          2010   
         10   The Lord of the Rings: The Fellowship of the Ring          2001   
         13               The Lord of the Rings: The Two Towers          2002   
...                                                         ...           ...   
Thriller 700                                    Wait Until Dark          1967   
Western  12                     Il buono, il brutto, il cattivo          1966   
         48                        Once Upon a Time in the West          1968   
         115                         Per qualche dollaro in più          1965   
         691                             The Outlaw Josey Wales          1976   

              Runtime  IMDB_Rating           Director              Star1  \
Genre                                                                      
Action   2        152          9.0  Christopher Nolan     Christian Bale   
         5        201          8.9      Peter Jackson        Elijah Wood   
         8        148          8.8  Christopher Nolan  Leonardo DiCaprio   
         10       178          8.8      Peter Jackson        Elijah Wood   
         13       179          8.7      Peter Jackson        Elijah Wood   
...               ...          ...                ...                ...   
Thriller 700      108          7.8      Terence Young     Audrey Hepburn   
Western  12       161          8.8       Sergio Leone     Clint Eastwood   
         48       165          8.5       Sergio Leone        Henry Fonda   
         115      132          8.3       Sergio Leone     Clint Eastwood   
         691      135          7.8     Clint Eastwood     Clint Eastwood   

              No_of_Votes        Gross  Metascore  norm_rating  
Genre                                                           
Action   2        2303232  534858444.0       84.0     1.000000  
         5        1642758  377845905.0       94.0     0.928571  
         8        2067042  292576195.0       74.0     0.857143  
         10       1661481  315544750.0       92.0     0.857143  
         13       1485555  342551365.0       87.0     0.785714  
...                   ...          ...        ...          ...  
Thriller 700        27733   17550741.0       81.0     1.000000  
Western  12        688390    6100000.0       90.0     1.000000  
         48        302844    5321508.0       80.0     0.700000  
         115       232772   15000000.0       74.0     0.500000  
         691        65659   31800000.0       69.0     0.000000  

[1000 rows x 10 columns]

In [50]:
#groupby on multiple columns 

#Ab main do coloumn ke basis pe groups banaunga to mujhe groupby() main lisst pass karna hoga 

duo=movies.groupby(["Director","Star1"])
duo.size()

duo.get_group(("Aamir Khan","Amole Gupte"))

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
65,Taare Zameen Par,2007,165,Drama,8.4,Aamir Khan,Amole Gupte,168895,1223869.0,NaN


In [ ]:
# find the most earning director-actor combo

duo["Gross"].max().sort_values(ascending=False).head(1)
#Suppose ek director and star ne 3 movie ki toh unme se gross ki value jiski zyada hogi sirf whi returnkarega lekinn mujhe to toal chahiye yni 3 movie ka total baaki  sabse compare hoga na too:-
duo["Gross"].sum().sort_values(ascending=False).head(1)

Director             Star1         
Aamir Khan           Amole Gupte         1223869.0
Aaron Sorkin         Eddie Redmayne    853090410.0
Abdellatif Kechiche  Léa Seydoux         2199675.0
Abhishek Chaubey     Shahid Kapoor     218428303.0
Abhishek Kapoor      Amit Sadh           1122527.0
                                          ...     
Zaza Urushadze       Lembit Ulfsak        144501.0
Zoya Akhtar          Hrithik Roshan      3108485.0
                     Vijay Varma         5566534.0
Çagan Irmak          Çetin Tekindor    461855363.0
Ömer Faruk Sorak     Cem Yilmaz        196206077.0
Name: Gross, Length: 898, dtype: float64

In [63]:
#actor- genre ka kounsa combo aisa hai jo avg metascore ke terms main best hai ?
duo1=movies.groupby(["Star1","Genre"])
duo1["Metascore"].mean().reset_index().sort_values("Metascore",ascending=False)

,Star1,Genre,Metascore
606,Peter O'Toole,Adventure,100.0
590,Orson Welles,Drama,100.0
77,Bertil Guve,Drama,100.0
230,Ellar Coltrane,Drama,100.0
329,Humphrey Bogart,Drama,100.0
...,...,...,...
807,William Holden,Drama,NaN
812,Won Bin,Action,NaN
815,Yash,Action,NaN
826,Çetin Tekindor,Drama,NaN


In [66]:
#Aggregate function on multiple groupby
numeric_cols=movies.select_dtypes(include="number").columns 
duo.mean(numeric_only=True)  

duo[numeric_cols].agg(["min","max","mean"])

Runtime             IMDB_Rating            \
                                       min  max   mean         min  max mean   
Director            Star1                                                      
Aamir Khan          Amole Gupte        165  165  165.0         8.4  8.4  8.4   
Aaron Sorkin        Eddie Redmayne     129  129  129.0         7.8  7.8  7.8   
Abdellatif Kechiche Léa Seydoux        180  180  180.0         7.7  7.7  7.7   
Abhishek Chaubey    Shahid Kapoor      148  148  148.0         7.8  7.8  7.8   
Abhishek Kapoor     Amit Sadh          130  130  130.0         7.7  7.7  7.7   
...                                    ...  ...    ...         ...  ...  ...   
Zaza Urushadze      Lembit Ulfsak       87   87   87.0         8.2  8.2  8.2   
Zoya Akhtar         Hrithik Roshan     155  155  155.0         8.1  8.1  8.1   
                    Vijay Varma        154  154  154.0         8.0  8.0  8.0   
Çagan Irmak         Çetin Tekindor     112  112  112.0         8.3  8.3  8.3   
Ömer Faruk Sorak    Cem Yilmaz         127  127  127.0         8.0  8.0  8.0   

                                   No_of_Votes                          Gross  \
                                           min     max      mean          min   
Director            Star1                                                       
Aamir Khan          Amole Gupte         168895  168895  168895.0    1223869.0   
Aaron Sorkin        Eddie Redmayne       89896   89896   89896.0  853090410.0   
Abdellatif Kechiche Léa Seydoux         138741  138741  138741.0    2199675.0   
Abhishek Chaubey    Shahid Kapoor        27175   27175   27175.0  218428303.0   
Abhishek Kapoor     Amit Sadh            32628   32628   32628.0    1122527.0   
...                                        ...     ...       ...          ...   
Zaza Urushadze      Lembit Ulfsak        40382   40382   40382.0     144501.0   
Zoya Akhtar         Hrithik Roshan       67927   67927   67927.0    3108485.0   
                    Vijay Varma          31886   31886   31886.0    5566534.0   
Çagan Irmak         Çetin Tekindor       78925   78925   78925.0  461855363.0   
Ömer Faruk Sorak    Cem Yilmaz           56960   56960   56960.0  196206077.0   

                                                             Metascore        \
                                            max         mean       min   max   
Director            Star1                                                      
Aamir Khan          Amole Gupte       1223869.0    1223869.0       NaN   NaN   
Aaron Sorkin        Eddie Redmayne  853090410.0  853090410.0      77.0  77.0   
Abdellatif Kechiche Léa Seydoux       2199675.0    2199675.0      89.0  89.0   
Abhishek Chaubey    Shahid Kapoor   218428303.0  218428303.0       NaN   NaN   
Abhishek Kapoor     Amit Sadh         1122527.0    1122527.0      40.0  40.0   
...                                         ...          ...       ...   ...   
Zaza Urushadze      Lembit Ulfsak      144501.0     144501.0      73.0  73.0   
Zoya Akhtar         Hrithik Roshan    3108485.0    3108485.0       NaN   NaN   
                    Vijay Varma       5566534.0    5566534.0      65.0  65.0   
Çagan Irmak         Çetin Tekindor  461855363.0  461855363.0       NaN   NaN   
Ömer Faruk Sorak    Cem Yilmaz      196206077.0  196206077.0       NaN   NaN   

                                          
                                    mean  
Director            Star1                 
Aamir Khan          Amole Gupte      NaN  
Aaron Sorkin        Eddie Redmayne  77.0  
Abdellatif Kechiche Léa Seydoux     89.0  
Abhishek Chaubey    Shahid Kapoor    NaN  
Abhishek Kapoor     Amit Sadh       40.0  
...                                  ...  
Zaza Urushadze      Lembit Ulfsak   73.0  
Zoya Akhtar         Hrithik Roshan   NaN  
                    Vijay Varma     65.0  
Çagan Irmak         Çetin Tekindor   NaN  
Ömer Faruk Sorak    Cem Yilmaz       NaN  

[898 rows x 15 columns]

In [ ]:
#Ipl ka ball by ball data hai 2022 tak ka jitne ipl khele gaye hai uske 
ipl = pd.read_csv(r"C:\Users\Asus\Downloads\deliveries.csv")
ipl

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,2,2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179073,11415,2,Chennai Super Kings,Mumbai Indians,20,2,RA Jadeja,SR Watson,SL Malinga,0,...,0,0,0,0,1,0,1,NaN,NaN,NaN
179074,11415,2,Chennai Super Kings,Mumbai Indians,20,3,SR Watson,RA Jadeja,SL Malinga,0,...,0,0,0,0,2,0,2,NaN,NaN,NaN
179075,11415,2,Chennai Super Kings,Mumbai Indians,20,4,SR Watson,RA Jadeja,SL Malinga,0,...,0,0,0,0,1,0,1,SR Watson,run out,KH Pandya
179076,11415,2,Chennai Super Kings,Mumbai Indians,20,5,SN Thakur,RA Jadeja,SL Malinga,0,...,0,0,0,0,2,0,2,NaN,NaN,NaN


In [ ]:
#top ten batsman in terms of runs
batter=ipl.groupby("batsman")
batter["batsman_runs"].sum().sort_values(ascending=False).head(10)

batsman
SK Raina          5651
V Kohli           5616
RG Sharma         5057
DA Warner         4975
S Dhawan          4876
CH Gayle          4873
RV Uthappa        4703
MS Dhoni          4691
AB de Villiers    4583
G Gambhir         4485
Name: total_runs, dtype: int64

In [82]:
#WOh batsman jisne ipl main sabse zyada six maare 

#sabse pehle to main sab whi balls nikalunga jisme six pade ho
sixes_only=ipl[ipl["batsman_runs"]==6]
sixes_only.groupby("batsman")["batsman"].count().sort_values(ascending=False).head(1).index[0]

'CH Gayle'

In [96]:
#finnd batsman with most numbers of fours and sixes in last five overs 
fours_sixes=ipl[((ipl["batsman_runs"]==6) | (ipl["batsman_runs"]==4)) & (ipl["over"]>15)]
batter1=fours_sixes.groupby("batsman")
batter1["batsman_runs"].count().sort_values(ascending=False).head(1).index[0]

'MS Dhoni'

In [103]:
#find kohli's record against all team (virat kohli ne har team ke againt matchh  khela hai to total kitne ruuns banaye )
record=(ipl[ipl["batsman"]=="V Kohli"]).groupby("bowling_team")
record["batsman_runs"].sum()


bowling_team
Chennai Super Kings        749
Deccan Chargers            306
Delhi Capitals              66
Delhi Daredevils           763
Gujarat Lions              283
Kings XI Punjab            636
Kochi Tuskers Kerala        50
Kolkata Knight Riders      675
Mumbai Indians             628
Pune Warriors              128
Rajasthan Royals           370
Rising Pune Supergiant      83
Rising Pune Supergiants    188
Sunrisers Hyderabad        509
Name: batsman_runs, dtype: int64

In [ ]:
# create a function that can return the highest score of any batsmans an input 
def highest(batsman):
  temp_df = ipl[ipl['batsman'] == batsman]
  return temp_df.groupby('match_id')['batsman_runs'].sum().sort_values(ascending=False).head(1).values[0]

highest('DA Warner')
